# Embedding Providers: Stable Contracts and Offline Evaluation

| Field | Value |
|---|---|
| Stage | Embeddings and indexes |
| Difficulty | Intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Provider swaps are index migrations. Keep model configuration explicit and compare quality, dimension, latency, and cost on one labeled dataset.

## 30-Second Summary

This notebook defines a provider-neutral embedding contract and compares two deterministic local implementations on the shared corpus. A hosted provider can replace either implementation without changing the evaluation harness.

## Why This Matters

Model names, dimensions, pricing, and APIs change. Binding notebook logic directly to one hosted client makes clean runs expensive and obscures whether a migration preserved retrieval behavior.

## Scope

| Covers | Does not cover |
|---|---|
| Provider contract, model versioning, deterministic local comparison, batching | Live paid calls, current price claims, production ANN index |


## Mental Model

```text
texts -> EmbeddingProvider(model, version) -> vectors(dim) -> versioned index
queries -> same provider/version -----------^
```


In [1]:
from hashlib import sha256
from pathlib import Path
from rag_101 import TfidfVectorizer, load_corpus, load_golden_questions

documents = load_corpus()
questions = [item for item in load_golden_questions() if item.relevant_doc_ids]
len(documents), len(questions)


(8, 7)

## How It Works

A provider must embed documents and queries into one compatible space and declare its identity and dimension. The index must record that identity; mixing versions requires re-embedding or separate indexes.


## Baseline

The baseline is the fitted TF-IDF representation from the foundation lesson. It is deterministic, corpus-specific, and strong on exact terms but weak on paraphrases.


In [2]:
class TfidfProvider:
    name = "local-tfidf-v1"
    def __init__(self, texts: list[str]):
        self.vectorizer = TfidfVectorizer().fit(texts)
        self.dimension = len(self.vectorizer.vocabulary)
    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.vectorizer.transform(texts)
    def embed_query(self, text: str) -> list[float]:
        return self.vectorizer.transform_one(text)

tfidf_provider = TfidfProvider([doc.content for doc in documents])
tfidf_provider.name, tfidf_provider.dimension


('local-tfidf-v1', 227)

## Technique Implementation

The comparison provider uses deterministic signed feature hashing. It has fixed dimension and no fitting step, which makes updates simple but introduces collisions. It is an interface example, not a semantic neural embedding.


In [3]:
import math, re

class HashingProvider:
    name = "local-hashing-v1"
    def __init__(self, dimension: int = 256): self.dimension = dimension
    def _embed(self, text: str) -> list[float]:
        vector = [0.0] * self.dimension
        for token in re.findall(r"[a-z0-9]+", text.lower()):
            digest = sha256(token.encode()).digest()
            index = int.from_bytes(digest[:4], "big") % self.dimension
            vector[index] += 1.0 if digest[4] % 2 == 0 else -1.0
        length = math.sqrt(sum(value * value for value in vector))
        return [value / length for value in vector] if length else vector
    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed(text) for text in texts]
    def embed_query(self, text: str) -> list[float]: return self._embed(text)

hashing_provider = HashingProvider()
hashing_provider.name, hashing_provider.dimension


('local-hashing-v1', 256)

## Controlled Experiment

Both providers embed the same corpus and seven answerable golden questions. We use the same cosine scorer and report hit rate@1; provider identity is the only changed component.


In [4]:
def dot(left, right): return sum(a * b for a, b in zip(left, right, strict=True))

def hit_rate(provider) -> float:
    vectors = provider.embed_documents([doc.content for doc in documents])
    hits = 0
    for question in questions:
        query_vector = provider.embed_query(question.question)
        best = max(zip(documents, vectors, strict=True), key=lambda pair: dot(query_vector, pair[1]))[0]
        hits += best.id in question.relevant_doc_ids
    return hits / len(questions)

results = {
    tfidf_provider.name: {"dimension": tfidf_provider.dimension, "hit_rate@1": hit_rate(tfidf_provider)},
    hashing_provider.name: {"dimension": hashing_provider.dimension, "hit_rate@1": hit_rate(hashing_provider)},
}
results


{'local-tfidf-v1': {'dimension': 227, 'hit_rate@1': 1.0},
 'local-hashing-v1': {'dimension': 256, 'hit_rate@1': 0.7142857142857143}}

## Evaluation

TF-IDF reaches **1.00 hit rate@1** on this exact-term-heavy golden set. Hashing is deterministic and fixed-width but may lose quality through collisions. Neither is a substitute for evaluating a true semantic model on paraphrases.


In [5]:
assert results["local-tfidf-v1"]["hit_rate@1"] == 1.0
assert all(len(vector) == hashing_provider.dimension for vector in hashing_provider.embed_documents(["a", "b"]))
assert hashing_provider.embed_query("repeatable") == hashing_provider.embed_query("repeatable")
assert tfidf_provider.name != hashing_provider.name
print("Provider-contract checks passed.")


Provider-contract checks passed.


## Decision Guide

| Need | Choice |
|---|---|
| Transparent lexical baseline | TF-IDF |
| Fixed local feature space | Hashing baseline |
| Semantic paraphrase retrieval | Evaluated local/hosted neural model |
| Provider migration | New index version plus parity/quality gate |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Query returns nonsense | Query/index model mismatch | Store and verify provider identity |
| Quality drops after upgrade | Model behavior changed | Run golden regression before cutover |
| Cost spike | Re-embedded unchanged content | Cache by content hash + model version |
| Clean run needs credentials | No offline default | Keep deterministic local path and tag paid cells |


## Production Notes

### Observability
Log provider/model/version, dimensions, batch size, latency, token/input volume, failures, and index version.

### Safety and Guardrails
Sending text to a provider is data transmission; classify/redact content and use approved regions and retention terms.

### Latency and Cost
Batch indexing, cache outputs, rate-limit retries, and obtain current pricing from the provider before budgeting.


## Practice

Implement a third provider behind the same methods and rerun the golden set without changing evaluation code.

## Recall

Toggle - Recall: Why is a provider swap an index migration?
Old and new vectors may have different dimensions and geometry.

Toggle - Recall: What stays fixed in a fair comparison?
Corpus, questions, scorer, cutoff, and labels.

## Sources

- [LangChain embeddings interface](https://python.langchain.com/docs/concepts/embedding_models/)
- [OpenAI embeddings guide](https://platform.openai.com/docs/guides/embeddings)
- Repository golden dataset

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the offline contract and regression harness | Run approved semantic providers with current cost/latency data |
